# v08 — Investigasi: pola "shakeout dulu baru lanjut ke arah benar"

**Latar belakang:** user amati chart live MT5 (7 Agustus 2026) — beberapa cluster sinyal BUY entry,
lalu harga sempat **turun dulu** (kemungkinan kena SL), baru **setelah itu** naik tajam ke arah yang
seharusnya profit. Arah sinyalnya benar, tapi timing entry-nya kena "jebakan" turun sesaat sebelum
breakout beneran (shakeout).

**Ini BUKAN** masalah yang sama dengan yang sudah diperbaiki sebelumnya:
- Bukan masalah v04 (SL fixed poin terlalu sempit utk rezim harga tinggi) — itu sudah diperbaiki di
  v06 (SL/TP ATR-relatif)
- Bukan solved by breakout confirmation — v07 sudah membuktikan menambah syarat breakout justru
  memperburuk hasil (menyaring terlalu banyak sinyal bagus)

**Pertanyaan notebook ini:** apakah pola "kena SL dulu, TAPI kalau ditunggu/SL lebih lebar sedikit
lagi, harga sebenarnya lanjut ke arah yang benar (profit)" ini POLA YANG SERING TERJADI di data
historis v06, atau cuma kebetulan di 2-3 contoh yang keliatan di chart? Analisis dulu sebelum ubah
parameter apa pun.

**Metodologi:** ambil semua trade v06 yang exit_reason=SL (534 dari 1664 trade). Untuk masing-masing,
cek pergerakan harga SETELAH candle SL exit — apakah harga akhirnya mencapai level TP originalnya
dalam N candle berikutnya (artinya: kalau SL sedikit lebih lebar, trade itu jadi WIN, bukan LOSS).

In [1]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent.parent
sys.path.append(str(PROJECT_ROOT))

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

STRATEGY_NAME = "m5_scalping"
VERSION = "v08"

PROCESSED_DIR = PROJECT_ROOT / "dataset" / "processed" / STRATEGY_NAME
EXPORT_DIR = PROJECT_ROOT / "dataset" / "exports" / STRATEGY_NAME / VERSION
EXPORT_DIR.mkdir(parents=True, exist_ok=True)

pd.set_option("display.width", 160)
plt.rcParams["figure.figsize"] = (14, 5)

## 1. Load trade log v06 (yang sudah ada, hasil backtest lengkap) + candle M5 mentah

In [2]:
trades = pd.read_csv(PROCESSED_DIR / "v06" / "trade_log_full.csv")
trades["entry_time"] = pd.to_datetime(trades["entry_time"])
trades["exit_time"] = pd.to_datetime(trades["exit_time"])

df_m5 = pd.read_csv(PROCESSED_DIR / "v01" / "xauusd_m5_full_indicators.csv")
df_m5["datetime"] = pd.to_datetime(df_m5["datetime"])
df_m5 = df_m5.sort_values("datetime").reset_index(drop=True)

sl_trades = trades[trades["exit_reason"] == "SL"].reset_index(drop=True)
print(f"Total trade v06: {len(trades)}")
print(f"Trade exit_reason=SL: {len(sl_trades)} ({len(sl_trades)/len(trades)*100:.1f}%)")
sl_trades[["entry_time", "direction", "entry_price", "sl_price", "tp_price", "pnl"]].head()

Total trade v06: 1664
Trade exit_reason=SL: 534 (32.1%)


,entry_time,direction,entry_price,sl_price,tp_price,pnl
0,2025-01-03 11:45:00+00:00,BUY,2657.868,2654.899311,2663.805379,-2.968689
1,2025-01-06 00:55:00+00:00,BUY,2643.365,2641.039244,2648.016512,-2.325756
2,2025-01-06 12:50:00+00:00,BUY,2642.878,2637.604394,2653.425211,-5.273606
3,2025-01-07 06:15:00+00:00,BUY,2645.248,2642.522379,2650.699241,-2.725621
4,2025-01-07 14:30:00+00:00,BUY,2661.835,2656.594315,2672.316370,-5.240685


## 2. Untuk tiap trade SL: apakah harga akhirnya sampai ke level TP originalnya setelahnya?

Cek N candle setelah exit SL — apakah harga (high utk BUY / low utk SELL) menyentuh `tp_price` asli.
Kalau iya dan SL-nya lebih lebar dikit, trade itu SEHARUSNYA jadi WIN, bukan LOSS — ini bukti
"shakeout lalu lanjut" beneran terjadi. Kalau harga malah terus turun / gak pernah balik, itu bukti
SL memang perlu (bukan cuma shakeout, arahnya beneran salah).

In [3]:
LOOKAHEAD_CANDLES = 24  # 2 jam ke depan setelah SL exit
m5_datetime_idx = df_m5.set_index("datetime")

results = []
for _, tr in sl_trades.iterrows():
    exit_time = tr["exit_time"]
    direction = tr["direction"]
    tp_price = tr["tp_price"]
    entry_price = tr["entry_price"]

    pos = df_m5["datetime"].searchsorted(exit_time)
    window = df_m5.iloc[pos: pos + LOOKAHEAD_CANDLES]
    if window.empty:
        continue

    if direction == "BUY":
        reached_tp = (window["high"] >= tp_price).any()
        max_favorable = window["high"].max() - entry_price
        max_adverse = entry_price - window["low"].min()
    else:
        reached_tp = (window["low"] <= tp_price).any()
        max_favorable = entry_price - window["low"].min()
        max_adverse = window["high"].max() - entry_price

    candles_to_tp = None
    if reached_tp:
        if direction == "BUY":
            hit_idx = window.index[window["high"] >= tp_price][0]
        else:
            hit_idx = window.index[window["low"] <= tp_price][0]
        candles_to_tp = hit_idx - window.index[0]

    results.append({
        "entry_time": tr["entry_time"], "direction": direction,
        "reached_original_tp_after_sl": bool(reached_tp),
        "candles_to_tp_after_sl": candles_to_tp,
        "max_favorable_after_sl": max_favorable,
        "max_adverse_after_sl": max_adverse,
        "sl_distance": tr["sl_points"],
        "tp_distance": tr["tp_points"],
    })

shakeout_df = pd.DataFrame(results)
pct_shakeout = shakeout_df["reached_original_tp_after_sl"].mean() * 100
print(f"Dari {len(shakeout_df)} trade yang kena SL, {shakeout_df['reached_original_tp_after_sl'].sum()} "
      f"({pct_shakeout:.1f}%) akhirnya harga SAMPAI ke level TP original dalam {LOOKAHEAD_CANDLES} candle (2 jam) berikutnya")
shakeout_df.head(10)

Dari 534 trade yang kena SL, 60 (11.2%) akhirnya harga SAMPAI ke level TP original dalam 24 candle (2 jam) berikutnya


,entry_time,direction,reached_original_tp_after_sl,candles_to_tp_after_sl,max_favorable_after_sl,max_adverse_after_sl,sl_distance,tp_distance
0,2025-01-03 11:45:00+00:00,BUY,False,NaN,-0.473,14.513,2.968689,5.937379
1,2025-01-06 00:55:00+00:00,BUY,False,NaN,3.910,3.780,2.325756,4.651512
2,2025-01-06 12:50:00+00:00,BUY,False,NaN,-3.343,28.513,5.273606,10.547211
3,2025-01-07 06:15:00+00:00,BUY,False,NaN,-0.623,7.303,2.725621,5.451241
4,2025-01-07 14:30:00+00:00,BUY,False,NaN,-3.440,19.800,5.240685,10.481370
5,2025-01-08 10:45:00+00:00,BUY,False,NaN,-0.897,8.321,2.320717,4.641434
6,2025-01-08 13:05:00+00:00,BUY,True,13.0,10.070,7.450,3.802719,7.605438
7,2025-01-09 08:30:00+00:00,BUY,False,NaN,1.450,5.491,3.217981,6.435963
8,2025-01-15 07:15:00+00:00,BUY,True,21.0,6.193,2.340,2.279426,4.558853
9,2025-01-15 21:00:00+00:00,BUY,False,NaN,1.230,4.127,2.280042,4.560083


## 3. Distribusi: berapa banyak SL yang "nyaris" (excess adverse kecil) vs "jauh" (excess besar)?

Kalau shakeout beneran pola dominan, `max_adverse_after_sl` (seberapa jauh harga masih turun SETELAH
SL kena, sebelum akhirnya balik) harusnya kecil/wajar relatif ke SL distance yang dipakai.

In [4]:
shakeout_df["adverse_vs_sl_ratio"] = shakeout_df["max_adverse_after_sl"] / shakeout_df["sl_distance"]

reached = shakeout_df[shakeout_df["reached_original_tp_after_sl"]]
not_reached = shakeout_df[~shakeout_df["reached_original_tp_after_sl"]]

print("=== Trade yang AKHIRNYA sampai TP setelah SL (kandidat 'shakeout') ===")
print(f"n = {len(reached)}")
print("candles_to_tp_after_sl (median):", reached["candles_to_tp_after_sl"].median())
print("adverse_vs_sl_ratio (median, seberapa jauh turun dulu relatif ke SL):", reached["adverse_vs_sl_ratio"].median())
print()
print("=== Trade yang TIDAK PERNAH sampai TP dalam 2 jam (SL memang perlu) ===")
print(f"n = {len(not_reached)}")
print("adverse_vs_sl_ratio (median):", not_reached["adverse_vs_sl_ratio"].median())

=== Trade yang AKHIRNYA sampai TP setelah SL (kandidat 'shakeout') ===
n = 60
candles_to_tp_after_sl (median): 14.5
adverse_vs_sl_ratio (median, seberapa jauh turun dulu relatif ke SL): 1.5038656377460469

=== Trade yang TIDAK PERNAH sampai TP dalam 2 jam (SL memang perlu) ===
n = 474
adverse_vs_sl_ratio (median): 2.1718239833454716


## 4. Simulasi: kalau SL dilebarkan (mis. 3x/4x ATR alih-alih 2x), berapa trade SL yang "terselamatkan" jadi WIN?

Ini cek KASAR (belum backtest penuh — cuma re-evaluasi trade yang SUDAH exit_reason=SL di v06) buat
estimasi awal apakah SL lebih lebar worth dicoba. `max_adverse_after_sl` dari cell 2 sudah termasuk
pergerakan SETELAH SL exit candle, jadi kita perlu re-cek dari SAAT ENTRY (bukan dari SL exit) --
gunakan `ind_atr` di trade log utk cek: kalau SL = 3xATR / 4xATR, apakah excursion max sebelum TP
tercapai tetap dalam batas itu?

In [5]:
def max_adverse_excursion_from_entry(entry_time, direction, entry_price, atr, lookahead=48):
    pos = df_m5["datetime"].searchsorted(entry_time)
    window = df_m5.iloc[pos: pos + lookahead]
    if window.empty:
        return None, None
    if direction == "BUY":
        mae = entry_price - window["low"].min()
    else:
        mae = window["high"].max() - entry_price
    return mae, mae / atr if atr > 0 else None


mae_results = []
for _, tr in sl_trades.iterrows():
    mae, mae_atr_ratio = max_adverse_excursion_from_entry(
        tr["entry_time"], tr["direction"], tr["entry_price"], tr["atr_at_entry"]
    )
    mae_results.append({"entry_time": tr["entry_time"], "mae": mae, "mae_atr_ratio": mae_atr_ratio})

mae_df = pd.DataFrame(mae_results)

for sl_mult_test in [2.0, 2.5, 3.0, 3.5, 4.0]:
    would_survive = (mae_df["mae_atr_ratio"] <= sl_mult_test).sum()
    pct = would_survive / len(mae_df) * 100
    print(f"SL={sl_mult_test}xATR -> {would_survive}/{len(mae_df)} ({pct:.1f}%) trade yang tadinya SL "
          f"di v06 (2.0x) TIDAK akan kena SL kalau SL dilebarkan segini (dlm 48 candle / 4 jam)")

SL=2.0xATR -> 0/534 (0.0%) trade yang tadinya SL di v06 (2.0x) TIDAK akan kena SL kalau SL dilebarkan segini (dlm 48 candle / 4 jam)


SL=2.5xATR -> 39/534 (7.3%) trade yang tadinya SL di v06 (2.0x) TIDAK akan kena SL kalau SL dilebarkan segini (dlm 48 candle / 4 jam)
SL=3.0xATR -> 92/534 (17.2%) trade yang tadinya SL di v06 (2.0x) TIDAK akan kena SL kalau SL dilebarkan segini (dlm 48 candle / 4 jam)
SL=3.5xATR -> 152/534 (28.5%) trade yang tadinya SL di v06 (2.0x) TIDAK akan kena SL kalau SL dilebarkan segini (dlm 48 candle / 4 jam)
SL=4.0xATR -> 207/534 (38.8%) trade yang tadinya SL di v06 (2.0x) TIDAK akan kena SL kalau SL dilebarkan segini (dlm 48 candle / 4 jam)


**Catatan penting soal cell di atas:** angka ini CUMA estimasi kasar "berapa trade yang tadinya SL
akan survive kalau SL dilebarkan" — TAPI melebarkan SL utk SEMUA trade juga akan mengubah nasib
trade yang SUDAH WIN/TIMEOUT sebelumnya (mereka jadi punya risk lebih besar per trade, dan sebagian
trade yang tadinya WIN bisa saja malah closed duluan di harga lain kalau exit window berubah). Untuk
kesimpulan valid harus full backtest ulang (seperti grid search v06), bukan cuma re-evaluasi trade
yang sudah SL. Ini cuma sinyal awal apakah worth dicoba.

## 5. Simpan hasil

In [6]:
shakeout_df.to_csv(EXPORT_DIR / "sl_trades_shakeout_analysis.csv", index=False)
mae_df.to_csv(EXPORT_DIR / "sl_trades_mae_analysis.csv", index=False)

with open(EXPORT_DIR / "summary.txt", "w") as f:
    f.write("v08 — Investigasi pola shakeout (SL kena dulu, lalu harga lanjut ke arah benar)\n\n")
    f.write(f"Total trade v06: {len(trades)}, exit_reason=SL: {len(sl_trades)} ({len(sl_trades)/len(trades)*100:.1f}%)\n")
    f.write(f"Dari trade SL, {pct_shakeout:.1f}% akhirnya harga sampai TP original dlm {LOOKAHEAD_CANDLES} candle (2 jam) berikutnya\n\n")
    f.write("Estimasi kasar 'trade SL yang akan survive kalau SL dilebarkan' (dlm 4 jam dari entry):\n")
    for sl_mult_test in [2.0, 2.5, 3.0, 3.5, 4.0]:
        would_survive = (mae_df["mae_atr_ratio"] <= sl_mult_test).sum()
        pct = would_survive / len(mae_df) * 100
        f.write(f"  SL={sl_mult_test}xATR: {would_survive}/{len(mae_df)} ({pct:.1f}%)\n")

print("Tersimpan ke:", EXPORT_DIR)
for f in sorted(EXPORT_DIR.glob("*")):
    print(" -", f.name)

Tersimpan ke: D:\Projects\robot-scalping\dataset\exports\m5_scalping\v08
 - sl_trades_mae_analysis.csv
 - sl_trades_shakeout_analysis.csv
 - summary.txt
